<a href="https://colab.research.google.com/github/maewhyxng2-prog/zyrquen-platform/blob/main/experiments_native_training_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ZYRQUEN Native V2 — Resume Training & Verification Lab

**สถานะปัจจุบัน:** `CANDIDATE — NOT PROMOTED`  
**โมเดล:** `PureNativeTransformerLMV2` + `PureContinualTrainerV2` (Native Python Core, ไม่ใช้ pretrained weights ภายนอก และไม่เรียก external AI API)

## คู่มือการใช้งาน & รายการตรวจสอบก่อนเริ่มฝึกต่อ (ZYRQUEN Pre-Training Checklist)
1. **ตรวจ Runtime / TPU (Cell 1):** รองรับทั้ง Google Colab TPU (`v5e-1 TPU` / `v6e-1 TPU` TorchTPU) และ CPU โดยไม่ติดตั้งแพ็กเกจเพิ่มอัตโนมัติ (Pure Native V2 ทำงานได้ทันทีบน CPU แม้ Colab ยังไม่ได้จัดสรร TPU)
2. **ตรวจพื้นที่เก็บ Checkpoint (`OUT_DIR`):** ไฟล์ `.json.gz` อยู่ที่ `/content/zyrquen-resume/checkpoints` (ไม่ถูก commit ลง GitHub)
3. **ตรวจ SHA-256 เทียบ Manifest ของ PR #56 (Cell 2):** ใช้ `verify_checkpoint_hash(step: int)` และตรวจ Step 4000, `v2_continued_step4025.json.gz`, Step 5000 baseline และชุด Step 5025–6000 (`v2_teacher_transfer_run5000_step6000.json.gz`)
4. **เลือก Checkpoint ตั้งต้นใน Cell 3:** ค่าเริ่มต้นโหลด Step 6000 (`v2_teacher_transfer_run5000_step6000.json.gz`) หรือเลือก `v2_continued_step4025.json.gz` / `latest_auto`
5. **ตรวจ Fresh-Runtime Reload & State:** ยืนยัน `global_step`, `m_state`, `v_state`, `replay_buffer`, `anchor_params` และ `importance_diag` หลังโหลด
6. **เทียบ Held-Out Transfer-Validation (Cell 4) และฝึกต่อถึง Step 7000 แบบ Non-Overwrite (Cell 5):** ตั้งชื่อ run ใหม่เสมอและสร้างไฟล์ `.sha256` ประกบทุกครั้ง หรือใช้ `!python3 tools/resume_training.py --step 6000 --target-step 7000 --train-file /path/to/train.jsonl`

> **คำเตือนการใช้งานใน Colab:** หากต้องการรันคำสั่ง CLI (`python3 tools/resume_training.py ...`) ใน code cell ให้ใส่เครื่องหมาย `!` นำหน้าเสมอ เช่น `!python3 tools/tpu_probe.py` หรือ `!python3 tools/resume_training.py --verify-only`

| Checkpoint | Global Step | Recorded SHA-256 |
|---|---:|---|
| `v2_continued_step4000.json.gz` | 4000 | `0ca27a632e85bc7fde764b9948ceb9ed16a53911d5773bd51b5cd7c722a5237d` |
| `v2_continued_step4025.json.gz` | 4025 | `f4b33c82a27a0a3f05b468d2ff1cf297746f7a3aa85ecbc03ff0093f34c75314` |
| `v2_teacher_transfer_step5000.json.gz` | 5000 | `76c85f56c615590906bee1f7893e22301f7be1d430130cd2b882e1921d9d72ed` |
| `v2_teacher_transfer_run5000_step6000.json.gz` | 6000 | `3b7f0ed4de259a0b1db2135caf193c074409cadb3a599ae4656cfd4b3dffbca7` |

In [ ]:
#@title 1. ตั้งค่าสภาวะแวดล้อมและตรวจ Colab TPU / TorchTPU Runtime
import hashlib
import json
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/zyrquen-platform")
OUT_DIR = Path("/content/zyrquen-resume/checkpoints")
REPO_URL = "https://github.com/maewhyxng2-prog/zyrquen-platform.git"

USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
if USE_GOOGLE_DRIVE and "google.colab" in sys.modules:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

if not REPO_DIR.exists():
    if (Path.cwd() / "python_core" / "v2" / "zyrquen_v2_pure.py").exists():
        REPO_DIR = Path.cwd()
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

PYTHON_CORE = REPO_DIR / "python_core"
for p in (str(PYTHON_CORE), str(REPO_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

from tools.eval_step6000_vs_step5000 import compare_step6000_vs_step5000, print_comparison_table
from tools.resume_training import resume_from_checkpoint
from tools.resume_v2_lab import (
    find_latest_checkpoint,
    load_texts_from_file,
    verify_and_reload,
)
from tools.tpu_probe import detect_runtime_accelerator, format_runtime_report
from v2.zyrquen_v2_pure import PureContinualTrainerV2
from v2.zyrquen_v2_trainer import ByteTokenizer
from verify_checkpoints import verify_checkpoints, print_report

runtime_info = detect_runtime_accelerator()
print(f"REPO_DIR : {REPO_DIR}")
print(f"OUT_DIR  : {OUT_DIR} (exists={OUT_DIR.exists()})")
print(format_runtime_report(runtime_info))

In [ ]:
#@title 2. ฟังก์ชัน verify_checkpoint_hash(step) และตรวจสอบ SHA-256 ตาม Manifest ใน PR #56
PR56_STEP_MANIFEST = {
    4000: ("v2_continued_step4000.json.gz", "0ca27a632e85bc7fde764b9948ceb9ed16a53911d5773bd51b5cd7c722a5237d"),
    4025: ("v2_continued_step4025.json.gz", "f4b33c82a27a0a3f05b468d2ff1cf297746f7a3aa85ecbc03ff0093f34c75314"),
    5000: ("v2_teacher_transfer_step5000.json.gz", "76c85f56c615590906bee1f7893e22301f7be1d430130cd2b882e1921d9d72ed"),
    5025: ("v2_teacher_transfer_run5000_step5025.json.gz", "1de745e95b02653a9dc4d9cd07a61c408553ac041256082e99fbbdf00e00e140"),
    5050: ("v2_teacher_transfer_run5000_step5050.json.gz", "3755c25a2e3c7e42e86349f3e283bfe1bc0e53cb836e4bb9b4d9cb5ce288dda0"),
    5075: ("v2_teacher_transfer_run5000_step5075.json.gz", "b289b0d6c588431b7a0443d6f5ef28cd99e8fe0dccfd70f4a560a17d5bd7b653"),
    5100: ("v2_teacher_transfer_run5000_step5100.json.gz", "ceef223f2751f2031a4a92516d4f42a108e3e387bd9f78c869504ea3954498fc"),
    5125: ("v2_teacher_transfer_run5000_step5125.json.gz", "4aa6ceab1132809647158b1b3ac0261c764217c744ec04c9e1df38611e3c0181"),
    5150: ("v2_teacher_transfer_run5000_step5150.json.gz", "53a5095ee2b74fd1094919141e5cbd4ff5f3521b28d7a6e9612f66813347ac72"),
    5175: ("v2_teacher_transfer_run5000_step5175.json.gz", "ff38aca43560457101fd5f0357f71767b42c717e86ba89366a3b665389399159"),
    5200: ("v2_teacher_transfer_run5000_step5200.json.gz", "801f1c064173b480bd75cc707d1795ef194b9673b5006835a8b6eb0a0ddfccc9"),
    5225: ("v2_teacher_transfer_run5000_step5225.json.gz", "499dfa5dc6e05dda0dc8ae04d160e645632b6dd146017c8c89e4733b08b4027e"),
    5250: ("v2_teacher_transfer_run5000_step5250.json.gz", "a185f847cec9ae4a73cd48998c612917adbf3d45e3ee50b4fb2e2612bfa869d3"),
    5275: ("v2_teacher_transfer_run5000_step5275.json.gz", "8b10636e775762cc1f4d7ee895baf5750f8850a1b7da9aaced90221dce050fc2"),
    5300: ("v2_teacher_transfer_run5000_step5300.json.gz", "285a473675c630dca1ff9b147d3e54e510a9233e83e83d3ecbad3859e9358df2"),
    5325: ("v2_teacher_transfer_run5000_step5325.json.gz", "56931475194b881be9e9d8c48007a764bf3a557e31702909af5bd8a21ed46294"),
    5350: ("v2_teacher_transfer_run5000_step5350.json.gz", "5fdc2aefcf9f77b54ca4ba36bcf498f0c7dfd37617f74be325cf76200fb65839"),
    5375: ("v2_teacher_transfer_run5000_step5375.json.gz", "a9c124e59116bf760aeab1273d6fe617b98c599258e2b9ed4c7450ef2a2e0b58"),
    5400: ("v2_teacher_transfer_run5000_step5400.json.gz", "8f93f6a1f996e755febf427c74779c35489a0b13e66c9dd6ff962ed4308effc7"),
    5425: ("v2_teacher_transfer_run5000_step5425.json.gz", "439a549e948c46a480a1aa1f465ba17dfc02e3a9b49c4e71b56e39977f75a09d"),
    5450: ("v2_teacher_transfer_run5000_step5450.json.gz", "001164a3fdf834a958dfad45028d941dba714590e039e7eed65e8ec246946fce"),
    5475: ("v2_teacher_transfer_run5000_step5475.json.gz", "eae1b6fa9798145bc4f77d566d2b1fdc8d0450addfa19e4cd80e5278a69691e7"),
    5500: ("v2_teacher_transfer_run5000_step5500.json.gz", "6b37593ef67a24729f8de2ec9349f7b3ccb2eb8a9e6054430f289a26070207a2"),
    5525: ("v2_teacher_transfer_run5000_step5525.json.gz", "bf1cc85e9e6221d8ec59bfcd182084735d4fa4792560d2c3ba89376d677d61fa"),
    5550: ("v2_teacher_transfer_run5000_step5550.json.gz", "883fd165bea22121e5d66813cc3357425ac7206e75126b79f1f7db0a697568c0"),
    5575: ("v2_teacher_transfer_run5000_step5575.json.gz", "b2f4e8575add3d5700b59c4d40b7bf1d64e10d7d27f1e5634a9ae6a296e21263"),
    5600: ("v2_teacher_transfer_run5000_step5600.json.gz", "8ba6e33f6eaa7d60911ee9ba3d3449a95b23b93711deddd27e368a2bed0b12b4"),
    5625: ("v2_teacher_transfer_run5000_step5625.json.gz", "8e52f2674096c054ccf6c846fcb3570232021ef9f11c58d9b685941bd0f7338e"),
    5650: ("v2_teacher_transfer_run5000_step5650.json.gz", "4fdce60b3ad836bd9a63b371d8c610a3efb135e14a94b3588c2cff8fad50c20a"),
    5675: ("v2_teacher_transfer_run5000_step5675.json.gz", "9fe00d7f6e73ff06cd18b65786b75109e7b3eacdcfb4f9a48ab415e3bb5441ad"),
    5700: ("v2_teacher_transfer_run5000_step5700.json.gz", "e3bfb82ce41c07ea009ac19ade9b4dadb11d26fb58e68fe10c3b9acc2d54900b"),
    5725: ("v2_teacher_transfer_run5000_step5725.json.gz", "adef12a38c715269bde139b88a20dba040501a7e05b8e03274f127ef48df36ed"),
    5750: ("v2_teacher_transfer_run5000_step5750.json.gz", "55549e8f24c4b4dd62e43a898dc035aa5fa9275b779395e98d06179c13b52e69"),
    5775: ("v2_teacher_transfer_run5000_step5775.json.gz", "cc4ccb15343175f32424fcca16b814996859ffb0c12d73412d0f06eab1b3f029"),
    5800: ("v2_teacher_transfer_run5000_step5800.json.gz", "9d17996720e8c72a3eeae61761aefd0495188911a83d152e89f3d8f42605c6f1"),
    5825: ("v2_teacher_transfer_run5000_step5825.json.gz", "ec0097e331167d81b13f92d6910962d68d7ba890cf21d925a6344b88310eec2f"),
    5850: ("v2_teacher_transfer_run5000_step5850.json.gz", "fba66bb041aae9cca57b43372e2159a94eefcbd682a3f40314a8bbd2adcbe5a8"),
    5875: ("v2_teacher_transfer_run5000_step5875.json.gz", "9ac2a73357d8a084e1a259c48a4b8eb3197415d9d29e585db2f5e4cf641f3a1f"),
    5900: ("v2_teacher_transfer_run5000_step5900.json.gz", "0b6d0b0e88f007ad663f30ce9a86f556a72d807814e5821e10c16f98558c431c"),
    5925: ("v2_teacher_transfer_run5000_step5925.json.gz", "6f078a01867a54545a24dede3fc4d982774c95a80fcb35111cfccf199f4034df"),
    5950: ("v2_teacher_transfer_run5000_step5950.json.gz", "afad144ed48bec461e9e22bfd8a4b1a7513cffa945a8e3a4f48ae98e2bf6c854"),
    5975: ("v2_teacher_transfer_run5000_step5975.json.gz", "abbc4c214d72a8f79339a89dc8c4b4ca97826a7df9e519d0f24c5bf4f7d6bd3a"),
    6000: ("v2_teacher_transfer_run5000_step6000.json.gz", "3b7f0ed4de259a0b1db2135caf193c074409cadb3a599ae4656cfd4b3dffbca7"),
}
PR56_SHA256_MANIFEST = {fname: sha for fname, sha in PR56_STEP_MANIFEST.values()}


def compute_file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def verify_checkpoint_against_pr56_manifest(ckpt_path: Path) -> str:
    if not ckpt_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    actual = compute_file_sha256(ckpt_path)
    expected = PR56_SHA256_MANIFEST.get(ckpt_path.name)
    if expected is None:
        sidecar = Path(f"{ckpt_path}.sha256")
        if sidecar.is_file():
            expected = sidecar.read_text(encoding="utf-8").strip().split()[0].lower()
    if expected is not None and actual != expected:
        raise ValueError(
            f"PR #56 SHA-256 mismatch for {ckpt_path.name}: actual={actual} expected={expected}"
        )
    return actual


def verify_checkpoint_hash(step: int, out_dir: Path = OUT_DIR) -> str:
    """อ่านไฟล์ checkpoint ตาม step และตรวจ SHA-256 กับ manifest ใน PR #56"""
    if step in PR56_STEP_MANIFEST:
        filename, expected = PR56_STEP_MANIFEST[step]
        ckpt_path = out_dir / filename
        if not ckpt_path.is_file():
            raise FileNotFoundError(f"Checkpoint step {step} not found: {ckpt_path}")
        actual = compute_file_sha256(ckpt_path)
        if actual != expected:
            raise ValueError(
                f"SHA-256 mismatch for step {step} ({filename}): actual={actual} expected={expected}"
            )
        return actual

    matches = sorted(p for p in out_dir.glob(f"*_step{step}.json.gz") if p.is_file())
    if not matches:
        raise FileNotFoundError(f"No checkpoint file found for step {step} in {out_dir}")
    return verify_checkpoint_against_pr56_manifest(matches[-1])


def verify_all_pr56_checkpoints(out_dir: Path) -> dict:
    assert out_dir.is_dir(), f"ไม่พบโฟลเดอร์ checkpoint: {out_dir}"
    overrides = {
        name: digest for name, digest in PR56_SHA256_MANIFEST.items() if (out_dir / name).is_file()
    }
    report = verify_checkpoints(out_dir, expected_overrides=overrides, allow_empty=False)
    print_report(report)
    if not report["ok"]:
        raise RuntimeError("พบ checkpoint ที่ SHA-256 ไม่ผ่านการตรวจสอบตาม PR #56 manifest")
    return report


_ = verify_all_pr56_checkpoints(OUT_DIR)

In [ ]:
#@title 3. โหลด Checkpoint ล่าสุด (Step 6000) และคืนค่า Optimizer/Trainer อย่างปลอดภัย
TARGET_CHECKPOINT = "v2_teacher_transfer_run5000_step6000.json.gz"  #@param ["v2_teacher_transfer_run5000_step6000.json.gz", "v2_continued_step4025.json.gz", "v2_teacher_transfer_step5000.json.gz", "latest_auto"]

STEP_LOOKUP = {fname: step for step, (fname, _) in PR56_STEP_MANIFEST.items()}
if TARGET_CHECKPOINT == "latest_auto":
    SELECTED_CKPT, EXPECTED_SHA = find_latest_checkpoint(OUT_DIR)
    verified_sha = verify_checkpoint_against_pr56_manifest(SELECTED_CKPT)
else:
    SELECTED_CKPT = OUT_DIR / TARGET_CHECKPOINT
    EXPECTED_SHA = PR56_SHA256_MANIFEST.get(TARGET_CHECKPOINT)
    if TARGET_CHECKPOINT in STEP_LOOKUP:
        verified_sha = verify_checkpoint_hash(STEP_LOOKUP[TARGET_CHECKPOINT], OUT_DIR)
    else:
        verified_sha = verify_checkpoint_against_pr56_manifest(SELECTED_CKPT)

reloaded = verify_and_reload(SELECTED_CKPT, expected_sha256=EXPECTED_SHA or verified_sha)
trainer: PureContinualTrainerV2 = reloaded["trainer"]
tokenizer: ByteTokenizer = reloaded["tokenizer"]

print("=== Pre-Training Checkpoint Reload Verified ===")
print(f"File                : {SELECTED_CKPT.name}")
print(f"SHA-256 (verified)  : {verified_sha}")
print(f"Size (bytes)        : {reloaded['size_bytes']:,}")
print(f"Embedded global_step: {reloaded['global_step']}")
print(f"Replay buffer items : {reloaded['replay_buffer_len']}")
print(f"Anchor / Importance : anchor={reloaded['has_anchor_params']}, importance={reloaded['has_importance_diag']}")

In [ ]:
#@title 4. ประเมิน Step 6000 เทียบ Step 5000 Baseline บน Held-Out Transfer-Validation
VAL_FILE = None  # ตัวอย่าง: Path("/content/zyrquen-resume/data/transfer_val.jsonl")
held_out_texts = load_texts_from_file(VAL_FILE) if VAL_FILE else []

if not held_out_texts:
    print("[INFO] ยังไม่ได้กำหนด VAL_FILE — ข้ามการคำนวณ metrics เพื่อไม่สร้างคะแนนจำลอง")
    print("       เมื่อมีไฟล์ชุด validation ให้ตั้งค่า VAL_FILE แล้วรันเซลล์นี้อีกครั้ง")
else:
    comparison = compare_step6000_vs_step5000(out_dir=OUT_DIR, val_texts=held_out_texts)
    print_comparison_table(comparison)
    eval_out = OUT_DIR / "step6000_vs_step5000_transfer_val.json"
    eval_out.write_text(json.dumps(comparison, indent=2), encoding="utf-8")
    print(f"Saved comparison artifact: {eval_out}")

In [ ]:
#@title 5. ฝึกโมเดลต่อจาก Step 6000 ไปยัง Step 7000 (Non-Overwrite + SHA-256 Sidecar)
TRAIN_FILE = None  # ตัวอย่าง: Path("/content/zyrquen-resume/data/transfer_train.jsonl")
train_texts = load_texts_from_file(TRAIN_FILE) if TRAIN_FILE else []

TARGET_STEP = 7000  #@param {type:"integer"}
CHUNK_STEPS = 25    #@param {type:"integer"}
BATCH_SIZE = 3      #@param {type:"integer"}
REPLAY_RATIO = 0.5  #@param {type:"number"}

if not train_texts:
    print("[WAIT] กรุณากำหนด TRAIN_FILE หรือใส่รายการข้อความใน train_texts ก่อนเริ่มฝึกต่อ")
elif trainer.global_step >= TARGET_STEP:
    print(f"[SKIP] global_step ปัจจุบัน ({trainer.global_step}) >= TARGET_STEP ({TARGET_STEP})")
else:
    start_step = trainer.global_step
    run_prefix = f"v2_teacher_transfer_run{start_step}"
    print(f"Resuming training: step {start_step} -> {TARGET_STEP} | train_examples={len(train_texts)}")

    while trainer.global_step < TARGET_STEP:
        step_budget = min(CHUNK_STEPS, TARGET_STEP - trainer.global_step)
        stats = trainer.train_steps(
            train_texts,
            steps=step_budget,
            batch_size=BATCH_SIZE,
            replay_ratio=REPLAY_RATIO,
        )
        out_ckpt = OUT_DIR / f"{run_prefix}_step{trainer.global_step}.json.gz"
        _, digest = trainer.save_checkpoint(out_ckpt)
        print(
            f"STEP {trainer.global_step} | loss={stats['final_loss']:.4f} | "
            f"file={out_ckpt.name} | sha256={digest}"
        )

In [ ]:
#@title 6. ตรวจ SHA-256 หลังฝึกจบและสรุปสถานะ Candidate
post_report = verify_checkpoints(OUT_DIR, allow_empty=False)
print(f"Directory check: total={post_report['total']} matched={post_report['matched']} failed={post_report['failed']}")
if held_out_texts:
    after_metrics = trainer.model.evaluate_texts(tokenizer, held_out_texts)
    print(f"Post-training held-out (step {trainer.global_step}): {after_metrics}")